# ME F321 Project — Fault Diagnosis of a Spacecraft Propulsion System

**Goal:** From 7 pressure signals (P1–P7, 1 kHz, 0–1200 ms) diagnose:

| Task | Question | Type | Training data used |
|---|---|---|---|
| 1 | Normal vs abnormal | Binary classification (+ novelty check) | all 177 cases |
| 2 | Bubble / solenoid-valve fault / **unknown** | Classification + novelty detection | 72 abnormal cases |
| 3 | Bubble location (BV1, BP1–BP7) | 8-class classification | 24 bubble cases |
| 4 | Which valve failed (SV1–SV4) | 4-class classification | 48 valve-fault cases |
| 5 | Opening ratio of the failed valve | Regression | 48 valve-fault cases |

**Key difficulty:** the test set contains Spacecraft-4, which never appears in training. So every model is validated with
**Leave-One-Spacecraft-Out (LOSO)** cross-validation: train on 2 spacecraft, test on the 3rd. This mimics the real test and
gives a much more honest score than a random split (section 9 shows the difference).

**Notebook map**
1. Setup & configuration  2. Load labels  3. Load signals + data quality  4. EDA (time, frequency, STFT)
5. Event detection  6. Feature extraction  7. Spacecraft normalisation  8. Feature selection + PCA
9–13. Tasks 1–5 (model comparison)  14. Final models + submission CSV  15. Bokeh dashboard  16. Notes for report

## 1. Setup & configuration
Edit `DATA_ROOT` to point at the folder you downloaded from Google Drive (the one containing the train/test folders,
`labels.xlsx` and `labels_spacecraft.xlsx`). The loader searches sub-folders recursively, so the exact layout does not matter.

In [ ]:
import re, warnings
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.signal import stft, welch

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif, mutual_info_regression
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)
# Classifiers (Labs 6, 7, 8)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              RandomForestRegressor, GradientBoostingRegressor)
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

DATA_ROOT = Path('./dataset')          # <-- EDIT: folder downloaded from Google Drive
OUT_DIR = Path('./outputs')            # figures, CSVs and dashboard are written here
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

FS = 1000                              # sampling rate (Hz) -> 1 sample = 1 ms
N_SAMPLES = 1200                       # 0-1200 ms
SENSORS = [f'P{i}' for i in range(1, 8)]
OPEN_MS = [100, 500, 900]              # nominal valve-open times (3 cycles)
CLOSE_MS = [300, 700, 1100]            # nominal valve-close times
BUBBLE_LOCS = ['BV1'] + [f'BP{i}' for i in range(1, 8)]
VALVES = ['SV1', 'SV2', 'SV3', 'SV4']
RS = 42                                # random_state used everywhere

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

## 2. Load and harmonise the label files
`labels.xlsx` describes each training case (case name, spacecraft, condition, valve opening ratios, bubble location).
Column names can vary, so `harmonize_labels()` detects them and converts everything into one clean table:
`case, spacecraft, condition (normal/bubble/valve), bubble_loc, valve, ratio`.

**Check the printed output** — if a column was not detected, rename it in the Excel file or adjust the keywords below.

In [ ]:
def find_file(pattern):
    hits = sorted(DATA_ROOT.rglob(pattern))
    if not hits:
        raise FileNotFoundError(f'No file matching {pattern} under {DATA_ROOT.resolve()}')
    return hits[0]

def _norm(s):
    return re.sub(r'[\s_\-#.]', '', str(s).lower())

def _find_col(df, keywords, exclude=()):
    for c in df.columns:
        n = _norm(c)
        if any(k in n for k in keywords) and not any(e in n for e in exclude):
            return c
    return None

def _truthy(v):
    return _norm(v) in {'yes', 'y', 'true', '1', '1.0', 'o', '○', '◯', 'x', '✓'}

def harmonize_labels(raw, is_train=True):
    df = raw.copy()
    case_col = _find_col(df, ['case'])
    sc_col = _find_col(df, ['spacecraft', 'sc', 'satellite'], exclude=['case'])
    out = pd.DataFrame({
        'case': df[case_col].astype(str).str.strip(),
        'spacecraft': df[sc_col].astype(str).str.extract(r'(\d+)')[0].astype(int),
    })
    print(f'case column -> {case_col!r}, spacecraft column -> {sc_col!r}')
    if not is_train:
        return out

    # --- valve opening ratios: columns SV1..SV4 (100 = healthy) ---
    sv_cols = {}
    for c in df.columns:
        m = re.fullmatch(r'sv([1-4])', _norm(c))
        if m:
            sv_cols[f'SV{m.group(1)}'] = c
    ratios = pd.DataFrame({v: pd.to_numeric(df[c].astype(str).str.replace('%', ''), errors='coerce')
                           for v, c in sv_cols.items()}) if sv_cols else None
    print('valve ratio columns ->', sv_cols)

    # --- bubble location: indicator columns (BV1, BP1..BP7) or a single text column ---
    loc_cols = {c: str(c).strip().upper() for c in df.columns if _norm(c).upper() in BUBBLE_LOCS}
    if loc_cols:
        bubble = df[list(loc_cols)].apply(
            lambda r: next((loc_cols[c] for c in loc_cols if _truthy(r[c])), np.nan), axis=1)
    else:
        bcol = _find_col(df, ['bubble', 'location'])
        bubble = df[bcol].astype(str).str.upper().str.extract(r'(BV1|BP[1-7])')[0] if bcol else pd.Series(np.nan, index=df.index)
    print('bubble-location columns ->', list(loc_cols) or 'text column')

    # --- condition ---
    cond_col = _find_col(df, ['condition', 'status', 'state', 'class'])
    def cond_from_text(t):
        t = _norm(t)
        if 'bubble' in t: return 'bubble'
        if 'valve' in t or 'sv' in t or 'fault' in t: return 'valve'
        if 'normal' in t: return 'normal'
        return np.nan
    cond = df[cond_col].map(cond_from_text) if cond_col else pd.Series(np.nan, index=df.index)
    # fill / cross-check from the other columns
    if ratios is not None:
        cond = cond.where(cond.notna() | ~(ratios < 100).any(axis=1), 'valve')
    cond = cond.where(cond.notna() | bubble.isna(), 'bubble').fillna('normal')
    out['condition'] = cond.values
    out['bubble_loc'] = np.where(out['condition'] == 'bubble', bubble, np.nan)

    if ratios is not None:
        faulty = ratios.idxmin(axis=1).where((ratios < 100).any(axis=1))
        out['valve'] = np.where(out['condition'] == 'valve', faulty, np.nan)
        out['ratio'] = np.where(out['condition'] == 'valve', ratios.min(axis=1), np.nan)
    else:
        out['valve'], out['ratio'] = np.nan, np.nan
    print('condition column ->', cond_col)
    return out

train_lab_raw = pd.read_excel(find_file('labels.xlsx'))
test_lab_raw = pd.read_excel(find_file('labels_spacecraft.xlsx'))
print('Raw training label columns:', list(train_lab_raw.columns))
display(train_lab_raw.head())

train_lab = harmonize_labels(train_lab_raw, is_train=True)
test_lab = harmonize_labels(test_lab_raw, is_train=False)
display(train_lab.head())
print(pd.crosstab(train_lab['condition'], train_lab['spacecraft'], margins=True))
print(pd.crosstab(train_lab['valve'], train_lab['ratio']))
print(test_lab['spacecraft'].value_counts())

## 3. Load the time-series + data-quality checks (Lab 3: missing values, outliers, duplicates)
Each CSV: column 0 = time, next 7 = P1–P7. Signals are stored as an array of shape `(n_cases, 7, 1200)`.

In [ ]:
CSV_INDEX = {}
for p in DATA_ROOT.rglob('*.csv'):
    CSV_INDEX.setdefault(p.stem.lower(), []).append(p)

def _digits(s):
    d = re.sub(r'\D', '', str(s)).lstrip('0')
    return d or '0'

def find_csv(case, sc, split):
    cands = CSV_INDEX.get(str(case).lower(), [])
    if not cands:  # e.g. label says "1" but file is "Case001.csv"
        cands = [p for k, ps in CSV_INDEX.items() for p in ps if _digits(k) == _digits(case)]
    if not cands:
        raise FileNotFoundError(f'CSV for case {case} not found')
    def score(p):
        s = _norm(p.as_posix())
        return 2 * (split in s) + (f'spacecraft{sc}' in s or f'sc{sc}' in s)
    return max(cands, key=score)

def read_case(path):
    df = pd.read_csv(path)
    if not all(isinstance(c, str) and re.search('[a-zA-Z]', c) for c in df.columns):
        df = pd.read_csv(path, header=None)           # file had no header row
    P = df.iloc[:, -7:].apply(pd.to_numeric, errors='coerce')
    n_nan = int(P.isna().sum().sum())
    P = P.interpolate(limit_direction='both')
    X = P.to_numpy().T
    n_raw = X.shape[1]
    if n_raw < N_SAMPLES:                              # pad with last value
        X = np.pad(X, ((0, 0), (0, N_SAMPLES - n_raw)), mode='edge')
    return X[:, :N_SAMPLES], n_nan, n_raw

def load_all(labels, split):
    sigs, qc = [], []
    for case, sc in zip(labels['case'], labels['spacecraft']):
        X, n_nan, n_raw = read_case(find_csv(case, sc, split))
        sigs.append(X); qc.append({'case': case, 'missing_values': n_nan, 'raw_length': n_raw})
    return np.stack(sigs), pd.DataFrame(qc)

X_train_sig, qc_train = load_all(train_lab, 'train')
X_test_sig, qc_test = load_all(test_lab, 'test')
print('Train signals:', X_train_sig.shape, '| Test signals:', X_test_sig.shape)

# Data-quality summary
for name, qc, S in [('train', qc_train, X_train_sig), ('test', qc_test, X_test_sig)]:
    flat = S.reshape(len(S), -1)
    dups = pd.DataFrame(flat).round(6).duplicated().sum()
    print(f"[{name}] missing values: {qc['missing_values'].sum()} | lengths: {sorted(qc['raw_length'].unique())} "
          f"| duplicate signals: {dups}")

# Outlier screening on per-case summary statistics (z-score > 3, as in Lab 3)
case_stats = pd.DataFrame({f'{s}_mean': X_train_sig[:, i].mean(1) for i, s in enumerate(SENSORS)})
Z = (case_stats - case_stats.mean()) / case_stats.std()
print('Cases with |z| > 3 on any sensor mean:', train_lab.loc[(Z.abs() > 3).any(axis=1), 'case'].tolist())
print(train_lab.loc[(Z.abs() > 3).any(axis=1), 'condition'].value_counts())
# NOTE: if the flagged cases are mostly real faults (valve/bubble), they are NOT data errors -> do not remove them.

## 4. Exploratory data analysis
### 4.1 Typical pressure profile (all sensors, one normal case)

In [ ]:
t = np.arange(N_SAMPLES)
idx_normal = np.where(train_lab['condition'] == 'normal')[0]
fig, ax = plt.subplots(figsize=(13, 4))
for i, s in enumerate(SENSORS):
    ax.plot(t, X_train_sig[idx_normal[0], i], lw=0.8, label=s)
for o, c in zip(OPEN_MS, CLOSE_MS):
    ax.axvspan(o, c, color='grey', alpha=0.08)
ax.set(xlabel='Time (ms)', ylabel='Pressure', title='Typical normal case (shaded = valves open)')
ax.legend(ncol=7, fontsize=8); savefig('eda_typical_profile'); plt.show()

### 4.2 Normal vs bubble vs valve fault — zoom on the first opening/closing transients
Bubbles change the speed of sound → frequency / damping / arrival-time changes. Valve faults reduce flow → smaller pressure drop and water-hammer amplitude.

In [ ]:
COND_COLORS = {'normal': 'tab:green', 'bubble': 'tab:blue', 'valve': 'tab:red'}
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
for ax, (i, s) in zip(axes.ravel(), enumerate(SENSORS)):
    for cond, col in COND_COLORS.items():
        sel = np.where(train_lab['condition'] == cond)[0]
        ax.plot(t[80:450], X_train_sig[sel, i, 80:450].mean(0), color=col, lw=1, label=cond)
    ax.set_title(s)
axes.ravel()[-1].axis('off'); axes[0, 0].legend()
fig.suptitle('Class-mean pressure, 80–450 ms'); plt.tight_layout(); savefig('eda_class_means'); plt.show()

### 4.3 Individual differences between spacecraft (domain shift)
Mean **normal** signal per spacecraft. Visible offsets here are why we use LOSO validation and spacecraft normalisation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for sc in sorted(train_lab['spacecraft'].unique()):
    sel = np.where((train_lab['condition'] == 'normal') & (train_lab['spacecraft'] == sc))[0]
    ax[0].plot(t, X_train_sig[sel, 0].mean(0), lw=0.8, label=f'SC{sc}')
    ax[1].plot(t[280:400], X_train_sig[sel, 0, 280:400].mean(0), lw=1, label=f'SC{sc}')
ax[0].set(title='P1, mean normal signal by spacecraft', xlabel='ms'); ax[1].set(title='Zoom: first valve closing', xlabel='ms')
ax[0].legend(); savefig('eda_spacecraft_shift'); plt.show()

### 4.4 Frequency domain: Welch PSD and STFT spectrogram (Lab 3 — data transformation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for cond, col in COND_COLORS.items():
    sel = np.where(train_lab['condition'] == cond)[0]
    seg = X_train_sig[sel, 0, 300:450]
    f, Pxx = welch(seg - seg.mean(1, keepdims=True), fs=FS, nperseg=128, axis=1)
    axes[0].semilogy(f, Pxx.mean(0), color=col, label=cond)
axes[0].set(xlim=(0, 300), xlabel='Frequency (Hz)', ylabel='PSD', title='P1 PSD after valve closing (300–450 ms)')
axes[0].legend()
f, tt, Zxx = stft(X_train_sig[idx_normal[0], 0] - X_train_sig[idx_normal[0], 0].mean(), fs=FS, nperseg=64, noverlap=56)
axes[1].pcolormesh(tt * 1000, f, np.abs(Zxx), shading='gouraud')
axes[1].set(ylim=(0, 300), xlabel='Time (ms)', ylabel='Frequency (Hz)', title='STFT of P1 (normal case)')
plt.tight_layout(); savefig('eda_psd_stft'); plt.show()

## 5. Event detection (align windows to actual valve timing)
Valve timing has ±1 ms uncertainty and differs between spacecraft. We locate each open/close event as the sample with the
largest total |dP/dt| within a few ms of its nominal time, and anchor all feature windows to these detected events.

In [ ]:
NOMINAL_EVENTS = np.array(sorted(OPEN_MS + CLOSE_MS))

def detect_events(X, search=(-6, 8)):
    d = np.abs(np.diff(X, axis=1)).sum(axis=0)
    ev = []
    for nom in NOMINAL_EVENTS:
        a, b = nom + search[0], nom + search[1]
        ev.append(a + int(np.argmax(d[a:b])))
    return np.array(ev)   # [open1, close1, open2, close2, open3, close3]

ev_train = np.array([detect_events(X) for X in X_train_sig])
offsets = pd.DataFrame(ev_train - NOMINAL_EVENTS, columns=['open1', 'close1', 'open2', 'close2', 'open3', 'close3'])
offsets['spacecraft'] = train_lab['spacecraft'].values
print('Mean detected offset from nominal (ms) per spacecraft:')
display(offsets.groupby('spacecraft').mean().round(2))

## 6. Feature extraction
Each case (7 × 1200 samples) is converted into a fixed-length feature vector. For each sensor and each of the 3 cycles we compute,
then **average over the cycles** (reduces noise):

| Group | Features | Physical meaning |
|---|---|---|
| Levels | baseline mean, flowing mean, pressure drop | flow rate → valve opening ratio |
| Opening transient (0–100 ms after open) | min, max, std, time of minimum | expansion wave |
| Closing transient (0–150 ms after close) | max, min, std, early/late std ratio (damping), time of peak | water hammer |
| Spectrum of closing / opening transients | dominant frequency & amplitude, spectral centroid, 3 band-energy ratios | acoustic modes → speed of sound → **bubbles** |
| Cross-sensor | lag (ms) between every sensor pair from cross-correlation (21 pairs) | wave travel time → **bubble location** |

In [ ]:
BANDS = [(5, 50), (50, 150), (150, 500)]

def _spectrum_feats(seg, nfft=2048):
    seg = seg - seg.mean()
    if np.allclose(seg, 0):
        return [np.nan, 0.0, np.nan] + [np.nan] * len(BANDS)
    S = np.abs(np.fft.rfft(seg * np.hanning(len(seg)), nfft))
    f = np.fft.rfftfreq(nfft, 1 / FS)
    m = f >= 5; f, S = f[m], S[m]
    P = S ** 2; tot = P.sum() + 1e-12
    i = int(np.argmax(S))
    return [f[i], S[i], (f * P).sum() / tot] + [P[(f >= lo) & (f < hi)].sum() / tot for lo, hi in BANDS]

def _xcorr_lag(a, b, maxlag=20):
    a = a - a.mean(); b = b - b.mean()
    cc = np.correlate(a, b, mode='full'); lags = np.arange(-len(b) + 1, len(a))
    m = np.abs(lags) <= maxlag; cc, lags = cc[m], lags[m]
    i = int(np.argmax(cc))
    if 0 < i < len(cc) - 1:                       # parabolic sub-sample refinement
        y0, y1, y2 = cc[i - 1], cc[i], cc[i + 1]; den = y0 - 2 * y1 + y2
        return lags[i] + (0.5 * (y0 - y2) / den if den != 0 else 0)
    return float(lags[i])

SPEC_NAMES = ['domfreq', 'domamp', 'centroid'] + [f'band{lo}_{hi}' for lo, hi in BANDS]

def extract_features(X):
    ev = detect_events(X)
    opens, closes = ev[0::2], ev[1::2]
    cyc = []
    for o, c in zip(opens, closes):
        f = {}
        for i, s in enumerate(SENSORS):
            x = X[i]
            base = x[max(o - 40, 0):o - 3].mean()
            flow = x[o + 120:c - 3].mean()
            op = x[o:o + 100]; cl = x[c:c + 150]
            f[f'{s}_base'] = base
            f[f'{s}_flow'] = flow
            f[f'{s}_drop'] = base - flow
            f[f'{s}_open_min'] = op.min() - base
            f[f'{s}_open_max'] = op.max() - base
            f[f'{s}_open_std'] = op.std()
            f[f'{s}_open_tmin'] = int(np.argmin(op[:60]))
            f[f'{s}_close_max'] = cl.max() - flow
            f[f'{s}_close_min'] = cl.min() - flow
            f[f'{s}_close_std'] = cl.std()
            f[f'{s}_close_decay'] = cl[:50].std() / (cl[50:].std() + 1e-9)
            f[f'{s}_close_tpeak'] = int(np.argmax(cl[:60]))
            for n, v in zip(SPEC_NAMES, _spectrum_feats(cl)): f[f'{s}_close_{n}'] = v
            for n, v in zip(SPEC_NAMES, _spectrum_feats(x[o:o + 110])): f[f'{s}_open_{n}'] = v
        for i in range(7):
            for j in range(i + 1, 7):
                f[f'lag_{SENSORS[i]}_{SENSORS[j]}'] = _xcorr_lag(X[i, c:c + 80], X[j, c:c + 80])
        f['open_duration'] = c - o
        cyc.append(f)
    return pd.DataFrame(cyc).mean()

F_train_raw = pd.DataFrame([extract_features(X) for X in X_train_sig])
F_test_raw = pd.DataFrame([extract_features(X) for X in X_test_sig])
# drop constant / all-NaN features
keep = F_train_raw.columns[(F_train_raw.std() > 1e-12) & F_train_raw.notna().any()]
F_train_raw, F_test_raw = F_train_raw[keep], F_test_raw[keep]
print('Feature matrix:', F_train_raw.shape)
display(F_train_raw.head())

## 7. Spacecraft normalisation (handling individual differences)
Subtract each spacecraft's median feature vector so that features describe *deviation from that spacecraft's typical
behaviour*. Training spacecraft use their training median; a spacecraft not seen in training (SC-4) uses the median of its own
test cases. We compare raw vs normalised features under LOSO in section 9 and keep whichever works better.

In [ ]:
def center_by_spacecraft(F_tr, sc_tr, F_te=None, sc_te=None):
    med = F_tr.groupby(sc_tr.values).median()
    Ftr = F_tr - med.loc[sc_tr.values].values
    if F_te is None:
        return Ftr
    Fte = F_te.copy()
    for sc in np.unique(sc_te):
        m = (sc_te.values == sc)
        ref = med.loc[sc] if sc in med.index else F_te[m].median()
        Fte.loc[m] = F_te[m].values - ref.values
    return Ftr, Fte

F_train_ctr, F_test_ctr = center_by_spacecraft(F_train_raw, train_lab['spacecraft'], F_test_raw, test_lab['spacecraft'])
groups = train_lab['spacecraft'].values

## 8. Feature analysis and selection (Lab 4: MI, correlation, ReliefF; Lab 3: PCA)
### 8.1 Correlation-based redundancy

In [ ]:
corr = F_train_ctr.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
redundant = [c for c in upper.columns if (upper[c] > 0.98).any()]
print(f'{len(redundant)} of {corr.shape[0]} features are >0.98 correlated with another feature')
plt.figure(figsize=(8, 7))
sns.heatmap(F_train_ctr.filter(like='P1_').corr(), cmap='coolwarm', center=0, xticklabels=True, yticklabels=True)
plt.title('Correlation among P1 features'); savefig('fs_corr_P1'); plt.show()

### 8.2 Mutual information per task + ReliefF

In [ ]:
def mi_rank(F, y, regression=False, top=15):
    Fi = F.fillna(F.median())
    fn = mutual_info_regression if regression else mutual_info_classif
    return pd.Series(fn(Fi, y, random_state=RS), index=F.columns).sort_values(ascending=False).head(top)

is_abn = train_lab['condition'] != 'normal'
is_bub = train_lab['condition'] == 'bubble'
is_val = train_lab['condition'] == 'valve'
MI = {
    'Task1 normal/abnormal': mi_rank(F_train_ctr, is_abn.astype(int)),
    'Task2 bubble/valve': mi_rank(F_train_ctr[is_abn], train_lab.loc[is_abn, 'condition']),
    'Task3 bubble location': mi_rank(F_train_ctr[is_bub], train_lab.loc[is_bub, 'bubble_loc']),
    'Task4 failed valve': mi_rank(F_train_ctr[is_val], train_lab.loc[is_val, 'valve']),
    'Task5 opening ratio': mi_rank(F_train_ctr[is_val], train_lab.loc[is_val, 'ratio'].astype(float), regression=True),
}
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, (k, s) in zip(axes, MI.items()):
    s[::-1].plot.barh(ax=ax); ax.set_title(k, fontsize=10)
plt.tight_layout(); savefig('fs_mutual_information'); plt.show()

try:
    from skrebate import ReliefF   # pip install skrebate
    Xs = StandardScaler().fit_transform(F_train_ctr.fillna(F_train_ctr.median()))
    rel = ReliefF(n_neighbors=10).fit(Xs, is_abn.astype(int).values)
    relief_scores = pd.Series(rel.feature_importances_, index=F_train_ctr.columns).sort_values(ascending=False)
    print('Top-10 ReliefF features (Task 1):'); print(relief_scores.head(10))
except ImportError:
    print('skrebate not installed -> skipping ReliefF (pip install skrebate)')

### 8.3 PCA visualisation (raw vs spacecraft-normalised)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
markers = {1: 'o', 2: 's', 3: '^'}
for ax, (title, F) in zip(axes, [('Raw features', F_train_raw), ('Spacecraft-normalised features', F_train_ctr)]):
    Z = PCA(2, random_state=RS).fit_transform(StandardScaler().fit_transform(F.fillna(F.median())))
    for cond, col in COND_COLORS.items():
        for sc, mk in markers.items():
            m = (train_lab['condition'] == cond).values & (groups == sc)
            ax.scatter(Z[m, 0], Z[m, 1], c=col, marker=mk, s=25, alpha=0.7,
                       label=f'{cond} SC{sc}' if ax is axes[0] else None)
    ax.set(title=title, xlabel='PC1', ylabel='PC2')
axes[0].legend(fontsize=7, ncol=3); plt.tight_layout(); savefig('pca_raw_vs_normalised'); plt.show()

## 9. Model zoo and evaluation helpers
Every model is wrapped in a `Pipeline`: median imputation → standardisation → (optional) SelectKBest(MI) → model.
Doing scaling and feature selection **inside** the pipeline prevents information leaking from the validation spacecraft.

Models compared (from the labs): Logistic Regression, Gaussian Naive Bayes, KNN, Decision Tree, Random Forest, Gradient
Boosting, linear SVM, RBF SVM, and a neural network (MLP).

In [ ]:
def get_classifiers():
    return {
        'LogReg':           LogisticRegression(max_iter=5000, C=1.0),
        'GaussianNB':       GaussianNB(),
        'KNN (k=3)':        KNeighborsClassifier(n_neighbors=3),
        'DecisionTree':     DecisionTreeClassifier(max_depth=5, random_state=RS),
        'RandomForest':     RandomForestClassifier(n_estimators=300, random_state=RS),
        'GradientBoosting': GradientBoostingClassifier(n_estimators=150, max_depth=2, random_state=RS),
        'SVM-Linear':       SVC(kernel='linear', C=1.0),
        'SVM-RBF':          SVC(kernel='rbf', C=10, gamma='scale'),
        'ANN (MLP)':        MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=3000, random_state=RS),
    }

def get_regressors():
    return {
        'Ridge':            Ridge(alpha=1.0),
        'KNN (k=3)':        KNeighborsRegressor(n_neighbors=3, weights='distance'),
        'DecisionTree':     DecisionTreeRegressor(max_depth=5, random_state=RS),
        'RandomForest':     RandomForestRegressor(n_estimators=300, random_state=RS),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=200, max_depth=2, random_state=RS),
        'SVR-RBF':          SVR(kernel='rbf', C=100, epsilon=1.0),
        'ANN (MLP)':        MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-2, max_iter=5000, random_state=RS),
    }

def make_pipe(model, k=None, regression=False):
    steps = [('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]
    if k is not None:
        score_fn = partial(mutual_info_regression if regression else mutual_info_classif, random_state=RS)
        steps.append(('select', SelectKBest(score_fn, k=k)))
    steps.append(('model', model))
    return Pipeline(steps)

LOSO = LeaveOneGroupOut()

def compare_classifiers(F, y, grp, k=None, cv=LOSO, task=''):
    rows, preds = [], {}
    k_eff = None if k is None else min(k, F.shape[1])
    for name, clf in get_classifiers().items():
        yp = cross_val_predict(make_pipe(clf, k_eff), F, y, groups=grp, cv=cv)
        preds[name] = yp
        rows.append({'Model': name, 'Accuracy': accuracy_score(y, yp),
                     'Balanced Acc': balanced_accuracy_score(y, yp),
                     'Macro F1': f1_score(y, yp, average='macro')})
    res = pd.DataFrame(rows).sort_values(['Macro F1', 'Balanced Acc'], ascending=False).reset_index(drop=True)
    if task:
        res.insert(0, 'Task', task)
    return res, preds

def compare_regressors(F, y, grp, k=None, cv=LOSO):
    rows, preds = [], {}
    k_eff = None if k is None else min(k, F.shape[1])
    for name, reg in get_regressors().items():
        yp = np.clip(cross_val_predict(make_pipe(reg, k_eff, regression=True), F, y, groups=grp, cv=cv), 0, 100)
        preds[name] = yp
        rows.append({'Model': name, 'MAE': mean_absolute_error(y, yp),
                     'RMSE': np.sqrt(mean_squared_error(y, yp)), 'R2': r2_score(y, yp)})
    return pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True), preds

def plot_comparison(res, metric, title, fname):
    ax = res.sort_values(metric).plot.barh(x='Model', y=metric, legend=False, figsize=(7, 4), color='tab:blue')
    ax.set(title=title, xlabel=metric); plt.tight_layout(); savefig(fname); plt.show()

def plot_cm(y, yp, labels, title, fname):
    ConfusionMatrixDisplay(confusion_matrix(y, yp, labels=labels), display_labels=labels).plot(cmap='Blues', values_format='d')
    plt.title(title); savefig(fname); plt.show()

RESULTS = {}   # collected for the dashboard

### 9.1 Which validation? Random split vs Leave-One-Spacecraft-Out
A random stratified 5-fold split mixes spacecraft between training and validation, which over-estimates performance on an
unseen spacecraft. Compare both on Task 1.

In [ ]:
y1 = is_abn.map({True: 'abnormal', False: 'normal'}).values
skf = StratifiedKFold(5, shuffle=True, random_state=RS)
r_rand, _ = compare_classifiers(F_train_raw, y1, groups, cv=skf)
r_loso, _ = compare_classifiers(F_train_raw, y1, groups, cv=LOSO)
cv_compare = r_rand[['Model', 'Macro F1']].merge(r_loso[['Model', 'Macro F1']], on='Model', suffixes=(' (random 5-fold)', ' (LOSO)'))
display(cv_compare)

### 9.2 Raw vs spacecraft-normalised features (LOSO, all tasks)
The feature set with the higher mean Macro-F1 across tasks 1–4 is used from here on.

In [ ]:
task_masks = {'T1': (slice(None), y1),
              'T2': (is_abn.values, train_lab.loc[is_abn, 'condition'].values),
              'T3': (is_bub.values, train_lab.loc[is_bub, 'bubble_loc'].values),
              'T4': (is_val.values, train_lab.loc[is_val, 'valve'].values)}
fs_rows = []
for fs_name, F in [('raw', F_train_raw), ('normalised', F_train_ctr)]:
    for tname, (m, yy) in task_masks.items():
        r, _ = compare_classifiers(F[m] if not isinstance(m, slice) else F, yy, groups[m])
        fs_rows.append({'features': fs_name, 'task': tname, 'best model': r.loc[0, 'Model'],
                        'best Macro F1': r.loc[0, 'Macro F1'], 'mean Macro F1 (all models)': r['Macro F1'].mean()})
fs_compare = pd.DataFrame(fs_rows)
display(fs_compare.pivot(index='task', columns='features', values=['best Macro F1', 'mean Macro F1 (all models)']).round(3))

USE_NORMALISED = (fs_compare.groupby('features')['best Macro F1'].mean().idxmax() == 'normalised')
F_tr = F_train_ctr if USE_NORMALISED else F_train_raw
F_te = F_test_ctr if USE_NORMALISED else F_test_raw
print('Using', 'spacecraft-normalised' if USE_NORMALISED else 'raw', 'features')

### 9.3 How many features? (SelectKBest with MI, inside the pipeline)
For each task, evaluate the best model with k ∈ {10, 20, 40, all}. Chosen k is stored in `BEST_K`.

In [ ]:
BEST_K = {}
k_grid = [10, 20, 40, None]
k_rows = []
for tname, (m, yy) in task_masks.items():
    Fm = F_tr if isinstance(m, slice) else F_tr[m]
    for k in k_grid:
        r, _ = compare_classifiers(Fm, yy, groups[m], k=k)
        k_rows.append({'task': tname, 'k': 'all' if k is None else k, 'best model': r.loc[0, 'Model'], 'Macro F1': r.loc[0, 'Macro F1']})
k_df = pd.DataFrame(k_rows)
display(k_df.pivot(index='task', columns='k', values='Macro F1').round(3))
for tname in task_masks:
    sub = k_df[k_df.task == tname].reset_index(drop=True)
    best = sub.loc[sub['Macro F1'].idxmax(), 'k']
    BEST_K[tname] = None if best == 'all' else int(best)
print('Chosen k per task:', BEST_K)

## 10. Task 1 — Normal vs abnormal
### 10.1 Supervised classifiers (LOSO)

In [ ]:
res1, preds1 = compare_classifiers(F_tr, y1, groups, k=BEST_K['T1'], task='Task 1')
display(res1)
plot_comparison(res1, 'Macro F1', 'Task 1: normal vs abnormal (LOSO)', 't1_comparison')
best1 = res1.loc[0, 'Model']
plot_cm(y1, preds1[best1], ['normal', 'abnormal'], f'Task 1 — {best1} (LOSO)', 't1_cm')
print(classification_report(y1, preds1[best1]))
RESULTS['Task 1'] = dict(res=res1, y=y1, yp=preds1[best1], labels=['normal', 'abnormal'], best=best1)

### 10.2 Adding a novelty detector (to also catch *unknown* anomalies)
A supervised classifier only learns the anomalies it has seen. We also fit a **normal-behaviour model**: distance from a
case to its k nearest *normal* training cases in PCA space. A case is flagged abnormal if the classifier says so **OR** its
distance exceeds the 97.5th percentile of LOSO distances of normal cases.

In [ ]:
class KNNNovelty:
    """Mean distance to the k nearest reference cases in standardised PCA space."""
    def __init__(self, k=3, var=0.95):
        self.k, self.var = k, var
    def fit(self, F):
        self.pre = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('pca', PCA(n_components=self.var, random_state=RS))]).fit(F)
        self.nn = NearestNeighbors(n_neighbors=min(self.k, len(F))).fit(self.pre.transform(F))
        return self
    def score(self, F):
        d, _ = self.nn.kneighbors(self.pre.transform(F))
        return d.mean(axis=1)

def loso_novelty_scores(F, ref_mask, grp, k=3):
    """For each spacecraft g: fit on reference cases of the other spacecraft, score every case of g."""
    s = np.zeros(len(F))
    for g in np.unique(grp):
        tr = ref_mask & (grp != g); te = grp == g
        s[te] = KNNNovelty(k).fit(F[tr]).score(F[te])
    return s

nov1 = loso_novelty_scores(F_tr, ~is_abn.values, groups)
THR1 = np.percentile(nov1[~is_abn.values], 97.5)
nov_pred = np.where(nov1 > THR1, 'abnormal', 'normal')
or_pred = np.where((preds1[best1] == 'abnormal') | (nov_pred == 'abnormal'), 'abnormal', 'normal')
t1_rules = pd.DataFrame([{'Rule': n, 'Accuracy': accuracy_score(y1, p), 'Balanced Acc': balanced_accuracy_score(y1, p),
                          'Macro F1': f1_score(y1, p, average='macro')}
                         for n, p in [(f'{best1} only', preds1[best1]), ('Novelty only', nov_pred), ('OR rule', or_pred)]])
display(t1_rules)
USE_NOVELTY_T1 = t1_rules.loc[2, 'Macro F1'] >= t1_rules.loc[0, 'Macro F1'] - 0.02   # keep OR rule unless it costs >2 pts
print('Use OR rule for Task 1:', USE_NOVELTY_T1)

plt.figure(figsize=(7, 3.5))
for cond, col in COND_COLORS.items():
    plt.hist(nov1[(train_lab['condition'] == cond).values], bins=25, alpha=0.6, color=col, label=cond)
plt.axvline(THR1, color='k', ls='--', label='threshold'); plt.legend(); plt.xlabel('Distance to normal cases')
plt.title('Task 1 novelty score (LOSO)'); savefig('t1_novelty'); plt.show()

## 11. Task 2 — Bubble vs solenoid-valve fault vs **unknown**
### 11.1 Known classes (LOSO, abnormal cases only)

In [ ]:
F2, y2, g2 = F_tr[is_abn.values], train_lab.loc[is_abn, 'condition'].values, groups[is_abn.values]
res2, preds2 = compare_classifiers(F2, y2, g2, k=BEST_K['T2'], task='Task 2')
display(res2)
plot_comparison(res2, 'Macro F1', 'Task 2: bubble vs valve fault (LOSO)', 't2_comparison')
best2 = res2.loc[0, 'Model']
plot_cm(y2, preds2[best2], ['bubble', 'valve'], f'Task 2 — {best2} (LOSO)', 't2_cm')
RESULTS['Task 2'] = dict(res=res2, y=y2, yp=preds2[best2], labels=['bubble', 'valve'], best=best2)

### 11.2 Unknown-anomaly detection
There are no unknown faults in training, so we **simulate** them: hide one known class, fit the novelty model on the other,
and check how many hidden-class cases get flagged as "unknown". The final rule: an abnormal case is labelled *unknown* if its
distance to all known abnormal training cases exceeds the 97.5th-percentile LOSO threshold.

In [ ]:
abn_ref = np.ones(is_abn.sum(), dtype=bool)
nov2 = loso_novelty_scores(F2, abn_ref, g2)
THR2 = np.percentile(nov2, 97.5)
print(f'Known abnormal cases wrongly flagged unknown (LOSO): {(nov2 > THR2).mean():.1%}')

sim_rows = []
for hidden in ['bubble', 'valve']:
    known = y2 != hidden
    s_known = loso_novelty_scores(F2, known, g2)          # LOSO scores relative to the known class
    thr = np.percentile(s_known[known], 97.5)
    s_hidden = KNNNovelty().fit(F2[known]).score(F2[~known])
    sim_rows.append({'Pretend unknown': hidden, 'Detected as unknown': f'{(s_hidden > thr).mean():.0%}',
                     'Known wrongly flagged': f'{(s_known[known] > thr).mean():.0%}'})
display(pd.DataFrame(sim_rows))

## 12. Task 3 — Bubble location (8 classes)
Only 24 training cases (3 per location, one per spacecraft) — so simple models (KNN, linear SVM, NB) often win here.

In [ ]:
F3, y3, g3 = F_tr[is_bub.values], train_lab.loc[is_bub, 'bubble_loc'].values, groups[is_bub.values]
res3, preds3 = compare_classifiers(F3, y3, g3, k=BEST_K['T3'], task='Task 3')
display(res3)
plot_comparison(res3, 'Macro F1', 'Task 3: bubble location (LOSO)', 't3_comparison')
best3 = res3.loc[0, 'Model']
plot_cm(y3, preds3[best3], BUBBLE_LOCS, f'Task 3 — {best3} (LOSO)', 't3_cm')
RESULTS['Task 3'] = dict(res=res3, y=y3, yp=preds3[best3], labels=BUBBLE_LOCS, best=best3)

## 13. Task 4 — Which solenoid valve failed (4 classes)

In [ ]:
F4, y4, g4 = F_tr[is_val.values], train_lab.loc[is_val, 'valve'].values, groups[is_val.values]
res4, preds4 = compare_classifiers(F4, y4, g4, k=BEST_K['T4'], task='Task 4')
display(res4)
plot_comparison(res4, 'Macro F1', 'Task 4: failed valve (LOSO)', 't4_comparison')
best4 = res4.loc[0, 'Model']
plot_cm(y4, preds4[best4], VALVES, f'Task 4 — {best4} (LOSO)', 't4_cm')
RESULTS['Task 4'] = dict(res=res4, y=y4, yp=preds4[best4], labels=VALVES, best=best4)

## 14. Task 5 — Opening ratio of the failed valve (regression)
Training ratios are discrete levels, while test ratios are random in 0–100 %. Models that interpolate smoothly
(Ridge, SVR, MLP) may generalise better than tree models, whose outputs are piecewise constant — check the scatter plot.
The failed valve (true in training, predicted in test) is added as one-hot features.

In [ ]:
def add_valve_onehot(F, valves):
    oh = pd.DataFrame({v: (np.asarray(valves) == v).astype(float) for v in VALVES}, index=F.index)
    return pd.concat([F, oh], axis=1)

y5 = train_lab.loc[is_val, 'ratio'].astype(float).values
F5 = add_valve_onehot(F4, y4)
res5, preds5 = compare_regressors(F5, y5, g4, k=None)
display(res5)
plot_comparison(res5, 'MAE', 'Task 5: opening ratio MAE (LOSO, lower is better)', 't5_comparison')
best5 = res5.loc[0, 'Model']
plt.figure(figsize=(5, 5))
plt.scatter(y5, preds5[best5], c=g4, cmap='viridis'); plt.plot([0, 100], [0, 100], 'k--')
plt.xlabel('True opening ratio (%)'); plt.ylabel('Predicted (%)'); plt.title(f'Task 5 — {best5} (LOSO)')
savefig('t5_scatter'); plt.show()
RESULTS['Task 5'] = dict(res=res5, y=y5, yp=preds5[best5], best=best5)

### 14.1 Summary of model comparison

In [ ]:
summary = pd.concat([res1, res2, res3, res4])
pivot = summary.pivot(index='Model', columns='Task', values='Macro F1').round(3)
pivot['Mean'] = pivot.mean(axis=1).round(3)
display(pivot.sort_values('Mean', ascending=False))
display(res5.round(2))
summary.to_csv(OUT_DIR / 'model_comparison_classification.csv', index=False)
res5.to_csv(OUT_DIR / 'model_comparison_regression.csv', index=False)

### 14.2 (Optional) Hyper-parameter tuning of a chosen model with GridSearchCV + LOSO (Lab 6)
Example for the Task 1 SVM; repeat for the final model of each task and justify the choice in the report.

In [ ]:
grid = GridSearchCV(make_pipe(SVC(kernel='rbf')), {'model__C': [0.1, 1, 10, 100], 'model__gamma': ['scale', 0.01, 0.001]},
                    cv=LOSO, scoring='f1_macro', n_jobs=-1)
grid.fit(F_tr, y1, groups=groups)
print('Best params:', grid.best_params_, '| LOSO macro-F1:', round(grid.best_score_, 3))

## 15. Final models → test predictions → submission CSV
By default the best LOSO model of each task is used. Override any choice in `FINAL` (e.g. prefer a simpler model when the
scores are tied — and justify it in the report).

In [ ]:
FINAL = {'T1': best1, 'T2': best2, 'T3': best3, 'T4': best4, 'T5': best5}
# FINAL['T3'] = 'SVM-Linear'    # example override
print('Final models:', FINAL)

clf_zoo, reg_zoo = get_classifiers(), get_regressors()
m1 = make_pipe(clf_zoo[FINAL['T1']], BEST_K['T1']).fit(F_tr, y1)
m2 = make_pipe(get_classifiers()[FINAL['T2']], BEST_K['T2']).fit(F2, y2)
m3 = make_pipe(get_classifiers()[FINAL['T3']], BEST_K['T3']).fit(F3, y3)
m4 = make_pipe(get_classifiers()[FINAL['T4']], BEST_K['T4']).fit(F4, y4)
m5 = make_pipe(reg_zoo[FINAL['T5']], None, regression=True).fit(F5, y5)
nov_normal = KNNNovelty().fit(F_tr[~is_abn.values])
nov_abnormal = KNNNovelty().fit(F2)

pred = test_lab[['case', 'spacecraft']].copy()
pred['task1'] = m1.predict(F_te)
pred['novelty_to_normal'] = nov_normal.score(F_te)
if USE_NOVELTY_T1:
    pred.loc[pred['novelty_to_normal'] > THR1, 'task1'] = 'abnormal'

abn = pred['task1'] == 'abnormal'
for col in ['task2', 'task3', 'task4']:
    pred[col] = pd.Series([None] * len(pred), dtype=object, index=pred.index)
pred['task5'] = np.nan
pred['novelty_to_known'] = np.nan
if abn.any():
    pred.loc[abn, 'task2'] = m2.predict(F_te[abn.values])
    pred.loc[abn, 'novelty_to_known'] = nov_abnormal.score(F_te[abn.values])
    pred.loc[abn & (pred['novelty_to_known'] > THR2), 'task2'] = 'unknown'
bub = pred['task2'] == 'bubble'
val = pred['task2'] == 'valve'
if bub.any():
    pred.loc[bub, 'task3'] = m3.predict(F_te[bub.values])
if val.any():
    pred.loc[val, 'task4'] = m4.predict(F_te[val.values])
    F5_te = add_valve_onehot(F_te[val.values], pred.loc[val, 'task4'].values)
    pred.loc[val, 'task5'] = np.clip(m5.predict(F5_te), 0, 100).round(1)

display(pred)
pred.to_csv(OUT_DIR / 'test_predictions_readable.csv', index=False)

# Numeric coding for submission — ADJUST to the official template when it is shared.
sub = pd.DataFrame({
    'Spacecraft No': pred['spacecraft'],
    'Case No': pred['case'],
    'task1': (pred['task1'] == 'abnormal').astype(int),                     # 0 normal, 1 abnormal
    'task2': pred['task2'].map({'bubble': 1, 'valve': 2, 'unknown': 3}),    # 1 bubble, 2 SV fault, 3 unknown
    'task3': pred['task3'].map({l: i + 1 for i, l in enumerate(BUBBLE_LOCS)}),  # 1=BV1, 2=BP1 ... 8=BP7
    'task4': pred['task4'].map({v: i + 1 for i, v in enumerate(VALVES)}),   # 1..4 = SV1..SV4
    'task5': pred['task5'],                                                 # opening ratio (%)
})
sub.to_csv(OUT_DIR / 'submission.csv', index=False)
print('Saved', OUT_DIR / 'submission.csv'); display(sub.head(10))

## 16. Dashboard (Bokeh, Lab 3) → `outputs/dashboard.html`
Five tabs: dataset overview, features, model comparison, Task 5 regression, test predictions. Open the HTML file in a browser.

In [ ]:
from bokeh.plotting import figure
from bokeh.models import (ColumnDataSource, Select, CustomJS, TabPanel, Tabs, DataTable, TableColumn,
                          HoverTool, Div, LinearColorMapper, ColorBar)
from bokeh.layouts import column, row
from bokeh.io import output_file, save
from bokeh.resources import INLINE   # embed BokehJS -> dashboard works offline
from bokeh.palettes import Category10, Blues256
from bokeh.transform import factor_cmap

conds = list(COND_COLORS)
BK = {'normal': '#2ca02c', 'bubble': '#1f77b4', 'valve': '#d62728'}   # hex colours for Bokeh
def df_table(df, width=900, height=280):
    df = df.copy().round(3) if hasattr(df, 'round') else df
    src = ColumnDataSource(df.astype(str))
    return DataTable(source=src, columns=[TableColumn(field=c, title=c) for c in df.columns], width=width, height=height)

# --- Tab 1: dataset ---
counts = train_lab.groupby(['spacecraft', 'condition']).size().reset_index(name='n')
x_fac = [(f'SC{s}', c) for s, c in zip(counts['spacecraft'], counts['condition'])]
from bokeh.models import FactorRange
p_counts = figure(x_range=FactorRange(*x_fac), height=300, width=600, title='Training cases per spacecraft and condition',
                  tools='hover', tooltips='@x: @n')
p_counts.vbar(x='x', top='n', width=0.85, source=ColumnDataSource(dict(x=x_fac, n=counts['n'])),
              fill_color=factor_cmap('x', palette=[BK[c] for c in conds], factors=conds, start=1, end=2))
p_counts.xaxis.major_label_orientation = 1.0

full = {'t': t}
for i, s in enumerate(SENSORS):
    for c in conds:
        full[f'{s}_{c}'] = X_train_sig[(train_lab['condition'] == c).values, i].mean(0)
full_src = ColumnDataSource(full)
show_src = ColumnDataSource({'t': t, **{f'y_{c}': full[f'P1_{c}'] for c in conds}})
p_sig = figure(height=330, width=900, title='Class-mean pressure signal', x_axis_label='Time (ms)', y_axis_label='Pressure')
for c in conds:
    p_sig.line('t', f'y_{c}', source=show_src, color=BK[c], legend_label=c, line_width=1.5)
sel_sensor = Select(title='Sensor', value='P1', options=SENSORS)
sel_sensor.js_on_change('value', CustomJS(args=dict(src=show_src, full=full_src, conds=conds), code="""
    const s = cb_obj.value;
    for (const c of conds) { src.data['y_' + c] = full.data[s + '_' + c]; }
    src.change.emit();"""))
info = Div(text=f"<h3>Dataset</h3><p>Training cases: {len(train_lab)} | Test cases: {len(test_lab)} | "
                f"Sensors: 7 (P1–P7) | 1 kHz, 1200 ms | Features extracted: {F_tr.shape[1]} | "
                f"Features used: {'spacecraft-normalised' if USE_NORMALISED else 'raw'}</p>", width=900)
tab1 = TabPanel(child=column(info, p_counts, sel_sensor, p_sig), title='Dataset')

# --- Tab 2: features ---
Zp = PCA(2, random_state=RS).fit_transform(StandardScaler().fit_transform(F_tr.fillna(F_tr.median())))
pca_src = ColumnDataSource(dict(pc1=Zp[:, 0], pc2=Zp[:, 1], cond=train_lab['condition'], sc=train_lab['spacecraft'].astype(str),
                                case=train_lab['case']))
p_pca = figure(height=420, width=520, title='PCA of features', tools='pan,wheel_zoom,reset,hover',
               tooltips='@case | @cond | SC@sc')
p_pca.scatter('pc1', 'pc2', source=pca_src, size=7, alpha=0.7, legend_field='cond',
              color=factor_cmap('cond', palette=[BK[c] for c in conds], factors=conds))
mi_name = Select(title='Mutual-information ranking for', value=list(MI)[0], options=list(MI))
mi_full = {k: (list(v.index[::-1]), list(v.values[::-1])) for k, v in MI.items()}
k0 = list(MI)[0]
mi_src = ColumnDataSource(dict(f=mi_full[k0][0], v=mi_full[k0][1]))
p_mi = figure(y_range=mi_full[k0][0], height=420, width=480, title='Top-15 features (MI)', tools='hover', tooltips='@f: @v{0.000}')
p_mi.hbar(y='f', right='v', height=0.7, source=mi_src)
mi_name.js_on_change('value', CustomJS(args=dict(src=mi_src, full=mi_full, yr=p_mi.y_range), code="""
    const d = full[cb_obj.value]; yr.factors = d[0]; src.data = {f: d[0], v: d[1]};"""))
tab2 = TabPanel(child=column(Div(text='<h3>Extracted features</h3>'), row(p_pca, column(mi_name, p_mi))), title='Features')

# --- Tab 3: models ---
plots = []
for task in ['Task 1', 'Task 2', 'Task 3', 'Task 4']:
    r = RESULTS[task]['res'].sort_values('Macro F1')
    p = figure(y_range=list(r['Model']), height=260, width=430, x_range=(0, 1.05), title=f'{task}: LOSO Macro-F1',
               tools='hover', tooltips='@Model: @{Macro F1}{0.000}')
    r = r.assign(color=['#ff7f0e' if m == RESULTS[task]['best'] else '#4c72b0' for m in r['Model']])
    p.hbar(y='Model', right='Macro F1', height=0.7, source=ColumnDataSource(r), color='color')
    plots.append(p)
def cm_plot(task):
    d = RESULTS[task]; labs = d['labels']
    cm = confusion_matrix(d['y'], d['yp'], labels=labs)
    xs, ys, vs = zip(*[(labs[j], labs[i], cm[i, j]) for i in range(len(labs)) for j in range(len(labs))])
    src = ColumnDataSource(dict(x=xs, y=ys, v=vs, txt=[str(v) for v in vs]))
    mapper = LinearColorMapper(palette=Blues256[::-1], low=0, high=max(cm.max(), 1))
    p = figure(x_range=labs, y_range=labs[::-1], height=300, width=330, title=f"{task} – {d['best']}",
               x_axis_label='Predicted', y_axis_label='True', tools='hover', tooltips='@y → @x: @v')
    p.rect('x', 'y', 1, 1, source=src, fill_color={'field': 'v', 'transform': mapper}, line_color='white')
    p.text('x', 'y', text='txt', source=src, text_align='center', text_baseline='middle')
    return p
tab3 = TabPanel(child=column(Div(text='<h3>Model comparison (Leave-One-Spacecraft-Out CV)</h3>'),
                             row(plots[0], plots[1]), row(plots[2], plots[3]),
                             row(*[cm_plot(t_) for t_ in ['Task 1', 'Task 2', 'Task 4']]), cm_plot('Task 3'),
                             df_table(pivot.reset_index())), title='Models')

# --- Tab 4: regression ---
r5 = RESULTS['Task 5']
p_reg = figure(height=400, width=420, title=f"Task 5 – {r5['best']} (LOSO)", x_axis_label='True %', y_axis_label='Predicted %',
               tools='hover', tooltips='true @x, pred @y{0.0}')
p_reg.scatter(x=list(r5['y']), y=list(r5['yp']), size=8, alpha=0.7)
p_reg.line(x=[0, 100], y=[0, 100], color='black', line_dash='dashed')
rr = res5.sort_values('MAE', ascending=False)
p_mae = figure(y_range=list(rr['Model']), height=400, width=420, title='MAE by model (lower is better)', tools='hover',
               tooltips='@Model: @MAE{0.00}')
p_mae.hbar(y='Model', right='MAE', height=0.7, source=ColumnDataSource(rr))
tab4 = TabPanel(child=column(row(p_reg, p_mae), df_table(res5, width=850, height=230)), title='Task 5 regression')

# --- Tab 5: test predictions ---
pc = pred['task2'].fillna('normal').value_counts()
p_pred = figure(x_range=list(pc.index), height=280, width=420, title='Predicted condition (test)', tools='hover', tooltips='@x: @top')
p_pred.vbar(x=list(pc.index), top=[int(v) for v in pc.values], width=0.7)
tab5 = TabPanel(child=column(p_pred, df_table(pred.drop(columns=[c for c in pred.columns if 'novelty' in c]), height=420)),
                title='Test predictions')

output_file(OUT_DIR / 'dashboard.html', title='MEF321 Propulsion PHM Dashboard')
save(Tabs(tabs=[tab1, tab2, tab3, tab4, tab5]), resources=INLINE)
print('Dashboard saved to', OUT_DIR / 'dashboard.html')

## 17. Notes for the report and presentation
* **Pre-processing:** data-quality checks (missing values, lengths, duplicates, z-score outliers); event detection to
  align windows; per-spacecraft normalisation to remove individual differences.
* **Feature engineering:** physics-motivated time-domain, frequency-domain (FFT/STFT) and cross-correlation features.
* **Feature selection:** correlation redundancy, mutual information, ReliefF; k chosen by LOSO CV inside a pipeline.
* **Models:** LogReg, Naive Bayes, KNN, Decision Tree, Random Forest, Gradient Boosting, linear/RBF SVM, MLP — compared on
  the same LOSO folds with accuracy, balanced accuracy and macro-F1 (MAE/RMSE/R² for Task 5).
* **Justify the final choice** with: LOSO score, stability across spacecraft, simplicity, and behaviour on the simulated
  unknown-fault test. Mention the random-split vs LOSO gap (section 9.1) as evidence of domain shift.
* **Limitations:** very few samples for Task 3 (3 per class); training opening ratios are discrete; SC-4 normalisation
  depends on the class mix of its test cases.